# Database initialization

In [4]:
from db_management_tools import DataLoader
dataset_dir = r"D:\AIC26"
data_loader = DataLoader(dataset_dir)

Dataset record:
873 videos
128567 keyframes



# Upload data

In [5]:
data_loader.create_and_fill_database(
    "aic2026_b1",
    upload_batch_size=256,
    upload_num_workers=16
)

Uploading dataset to Qdrant: 100%|██████████| 128567/128567 [13:47<00:00, 155.43it/s]


# Query initialization

In [1]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model
from peripheral_tools import NotebookDisplay

image_text_feature_extractor = SigLIP2Model(name="google/siglip2-giant-opt-patch16-384", device="cuda")
text_text_feature_extractor = BGEM3Model(name="BAAI/bge-m3", device="cuda")

query_manager = QueryManager(
    image_text_feature_extractor=image_text_feature_extractor,
    text_text_feature_extractor=text_text_feature_extractor
)
display = NotebookDisplay(315, 5)

[INFO] Loading quantized google/siglip2-giant-opt-patch16-384
[INFO] google/siglip2-giant-opt-patch16-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


In [3]:
query = "Một người áo đỏ, đội nón màu trắng, đang lấy nước rưới vào mặt của mình. Khung hình có hai người đi xe đạp, người mặc áo xanh đậm đang đuổi theo người mặc áo đen phối cam."
limit = 100

# Fast search
# list_vectors = ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
# points = query_manager.search(
#     "aic2026_b1",
#     query=query,
#     search_limit=limit,
#     using_vectors=list_vectors[1:3]
# )

# Agg search v1
points = query_manager.aggregated_search(
    "aic2026_b1",
    query=query,
    search_limit=limit,
    aggregation_weight=0.5
)

# Agg search v2
# points = query_manager.aggregated_search_v2(
#     "aic2026_b1",
#     query=query,
#     search_limit=limit,
#     aggregation_neighbor_limit=10,
#     time_distance_threshold=15,
#     use_time_distance=True,
#     entropy_based_bonus=False,
#     agg_z_score_sigmoid_norm=False,
#     aggregation_weight=0.5,
#     num_workers=32,
#     logging=True,
# )

# print("\nList of candidates:")
# for point in points: print(f"{point.payload["vid_id"]}_{point.payload["frame_id"]}")
display.display(points)

['L23_V019_105', 'L21_V022_364', 'L21_V025_130', 'L22_V004_595', 'L21_V025_129']


['L23_V019_104', 'L23_V019_107', 'L21_V002_304', 'L23_V019_113', 'L23_V019_106']


['L21_V029_350', 'L22_V021_544', 'L22_V021_545', 'L22_V021_543', 'L22_V021_548']


['L21_V025_131', 'L22_V021_547', 'L22_V021_546', 'L22_V021_549', 'L21_V025_127']


['L23_V019_111', 'L22_V021_539', 'L23_V019_108', 'L23_V017_037', 'L22_V021_550']


...

['L23_V017_044', 'L22_V021_541', 'L21_V005_311', 'L23_V010_002', 'L21_V025_122']


['L23_V019_083', 'L23_V019_085', 'L23_V019_084', 'L21_V026_499', 'L23_V019_087']


['L21_V027_045', 'L22_V008_173', 'L21_V018_399', 'L21_V023_172', 'L23_V019_086']


['L22_V024_404', 'L23_V010_003', 'L21_V029_549', 'L22_V004_023', 'L21_V027_520']


['L22_V016_469', 'L22_V024_401', 'L23_V010_004', 'L21_V009_116', 'L22_V021_538']


# Benchmark

In [ ]:
from peripheral_tools import Benchmarker, BenchmarkerV2

test_settings = [
    # {
    #     "using_vectors": ["siglip2_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_sparse"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    # },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
]

benchmarker = BenchmarkerV2(query_manager)
res = benchmarker.do_benchmark(
    "aic26_mini",
    test_questions=r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\question_bank\questions_with_answers.csv",
    test_settings=test_settings,
    save_record=False
)